In [1]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from time import sleep
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
# Project configuration

START_DATE = "20160101"
END_DATE = "20251231"

BASE_URL = (
    "https://power.larc.nasa.gov/api/temporal/daily/point"
)

PARAMETERS = [
    "ALLSKY_SFC_SW_DWN",
    "T2M",
    "T2M_MAX",
    "T2M_MIN",
    "RH2M",
    "WS10M",
    "PRECTOTCORR"
]

RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project configuration completed!")

Project configuration completed!


In [3]:
locations = [
    {
        "city": "Lucknow",
        "state": "Uttar Pradesh",
        "latitude": 26.8467,
        "longitude": 80.9462
    },
    {
        "city": "Dehradun",
        "state": "Uttarakhand",
        "latitude": 30.3165,
        "longitude": 78.0322
    },
    {
        "city": "New Delhi",
        "state": "Delhi",
        "latitude": 28.6139,
        "longitude": 77.2090
    },
    {
        "city": "Gurugram",
        "state": "Haryana",
        "latitude": 28.4595,
        "longitude": 77.0266
    },
    {
        "city": "Ludhiana",
        "state": "Punjab",
        "latitude": 30.9010,
        "longitude": 75.8573
    },
    {
        "city": "Shimla",
        "state": "Himachal Pradesh",
        "latitude": 31.1048,
        "longitude": 77.1734
    },
    {
        "city": "Jaipur",
        "state": "Rajasthan",
        "latitude": 26.9124,
        "longitude": 75.7873
    },
    {
        "city": "Srinagar",
        "state": "Jammu & Kashmir",
        "latitude": 34.0837,
        "longitude": 74.7973
    },
    {
        "city": "Patna",
        "state": "Bihar",
        "latitude": 25.5941,
        "longitude": 85.1376
    },
    {
        "city": "Bhopal",
        "state": "Madhya Pradesh",
        "latitude": 23.2599,
        "longitude": 77.4126
    }
]

locations_df = pd.DataFrame(locations)

display(locations_df)

,city,state,latitude,longitude
0,Lucknow,Uttar Pradesh,26.8467,80.9462
1,Dehradun,Uttarakhand,30.3165,78.0322
2,New Delhi,Delhi,28.6139,77.2090
3,Gurugram,Haryana,28.4595,77.0266
4,Ludhiana,Punjab,30.9010,75.8573
5,Shimla,Himachal Pradesh,31.1048,77.1734
6,Jaipur,Rajasthan,26.9124,75.7873
7,Srinagar,Jammu & Kashmir,34.0837,74.7973
8,Patna,Bihar,25.5941,85.1376
9,Bhopal,Madhya Pradesh,23.2599,77.4126


In [4]:
session = requests.Session()

retries = Retry(
    total=4,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"]
)

adapter = HTTPAdapter(max_retries=retries)

session.mount("https://", adapter)


def collect_solar_data(location):

    params = {
        "parameters": ",".join(PARAMETERS),
        "community": "RE",
        "longitude": location["longitude"],
        "latitude": location["latitude"],
        "start": START_DATE,
        "end": END_DATE,
        "format": "JSON",
        "time-standard": "LST"
    }

    response = session.get(
        BASE_URL,
        params=params,
        timeout=90
    )

    response.raise_for_status()

    data = response.json()

    if "properties" not in data:
        raise ValueError(f"Unexpected API response: {data}")

    parameter_data = data["properties"]["parameter"]

    df = pd.DataFrame(parameter_data)

    df.index = pd.to_datetime(
        df.index,
        format="%Y%m%d"
    )

    df.index.name = "DATE"

    df = df.reset_index()

    # Add geographical information
    df["CITY"] = location["city"]
    df["STATE"] = location["state"]
    df["LATITUDE"] = location["latitude"]
    df["LONGITUDE"] = location["longitude"]

    # Convert numerical columns
    numeric_columns = [
        col for col in PARAMETERS
    ]

    df[numeric_columns] = df[numeric_columns].apply(
        pd.to_numeric,
        errors="coerce"
    )

    # Handle NASA missing-value codes
    df[numeric_columns] = df[numeric_columns].mask(
        df[numeric_columns] <= -900
    )

    return df

In [5]:
lucknow_data = collect_solar_data(locations[0])

print("NASA POWER API connection successful!")

print("\nDataset preview:")
display(lucknow_data.head())

print("\nDataset dimensions:")
print(lucknow_data.shape)

print("\nMissing values:")
display(lucknow_data.isnull().sum())

print("\nDate range:")
print(lucknow_data["DATE"].min())
print(lucknow_data["DATE"].max())

NASA POWER API connection successful!

Dataset preview:


,DATE,ALLSKY_SFC_SW_DWN,T2M,T2M_MAX,T2M_MIN,RH2M,WS10M,PRECTOTCORR,CITY,STATE,LATITUDE,LONGITUDE
0,2016-01-01,3.9835,18.81,27.00,12.49,22.67,0.94,0.0,Lucknow,Uttar Pradesh,26.8467,80.9462
1,2016-01-02,3.4915,17.82,26.83,11.12,20.59,2.34,0.0,Lucknow,Uttar Pradesh,26.8467,80.9462
2,2016-01-03,3.1678,15.85,25.49,8.49,21.67,2.73,0.0,Lucknow,Uttar Pradesh,26.8467,80.9462
3,2016-01-04,2.9002,17.72,25.33,10.15,20.52,1.24,0.0,Lucknow,Uttar Pradesh,26.8467,80.9462
4,2016-01-05,2.8135,19.79,27.32,12.33,22.22,1.81,0.0,Lucknow,Uttar Pradesh,26.8467,80.9462



Dataset dimensions:
(3653, 12)

Missing values:


DATE                 0
ALLSKY_SFC_SW_DWN    0
T2M                  0
T2M_MAX              0
T2M_MIN              0
RH2M                 0
WS10M                0
PRECTOTCORR          0
CITY                 0
STATE                0
LATITUDE             0
LONGITUDE            0
dtype: int64


Date range:
2016-01-01 00:00:00
2025-12-31 00:00:00


In [6]:
all_city_data = []

for i, location in enumerate(locations, start=1):

    print(
        f"Downloading {i}/10: {location['city']}..."
    )

    try:
        city_data = collect_solar_data(location)

        filename = (
            f"{location['city'].lower().replace(' ', '_')}"
            "_solar_data.csv"
        )

        city_data.to_csv(
            RAW_DIR / filename,
            index=False
        )

        all_city_data.append(city_data)

        print(
            f"Success: {len(city_data)} records downloaded"
        )

    except Exception as error:
        print(
            f"Failed: {location['city']} | {error}"
        )

    sleep(1)

print("\nData collection process completed!")

Success: 3653 records downloaded
Success: 3653 records downloaded
Success: 3653 records downloaded
Success: 3653 records downloaded
Success: 3653 records downloaded
Success: 3653 records downloaded
Success: 3653 records downloaded
Success: 3653 records downloaded
Success: 3653 records downloaded
Success: 3653 records downloaded

Data collection process completed!


In [7]:
if len(all_city_data) != len(locations):
    raise RuntimeError(
        "Some cities failed to download. "
        "Check the errors before proceeding."
    )

combined_data = pd.concat(
    all_city_data,
    ignore_index=True
)

combined_data = combined_data.sort_values(
    ["CITY", "DATE"]
).reset_index(drop=True)

combined_data.to_csv(
    RAW_DIR / "india_solar_combined.csv",
    index=False
)

print("Combined dataset saved successfully!")

print("Total records:", len(combined_data))
print("Total cities:", combined_data["CITY"].nunique())

display(combined_data.head(10))

Combined dataset saved successfully!
Total records: 36530
Total cities: 10


,DATE,ALLSKY_SFC_SW_DWN,T2M,T2M_MAX,T2M_MIN,RH2M,WS10M,PRECTOTCORR,CITY,STATE,LATITUDE,LONGITUDE
0,2016-01-01,4.4923,19.93,29.97,11.94,32.76,2.64,0.00,Bhopal,Madhya Pradesh,23.2599,77.4126
1,2016-01-02,4.1618,18.99,28.65,11.18,30.77,2.54,0.00,Bhopal,Madhya Pradesh,23.2599,77.4126
2,2016-01-03,4.1638,18.80,27.93,11.07,30.76,2.23,0.00,Bhopal,Madhya Pradesh,23.2599,77.4126
3,2016-01-04,4.4002,19.87,28.74,10.50,26.03,1.96,0.00,Bhopal,Madhya Pradesh,23.2599,77.4126
4,2016-01-05,4.5458,20.31,29.18,11.27,25.77,1.64,0.00,Bhopal,Madhya Pradesh,23.2599,77.4126
5,2016-01-06,4.5449,21.18,29.79,12.61,25.23,1.97,0.00,Bhopal,Madhya Pradesh,23.2599,77.4126
6,2016-01-07,4.4626,20.73,29.49,13.30,33.78,2.23,0.00,Bhopal,Madhya Pradesh,23.2599,77.4126
7,2016-01-08,4.3181,20.82,28.90,15.00,35.47,2.35,0.35,Bhopal,Madhya Pradesh,23.2599,77.4126
8,2016-01-09,4.5170,18.48,27.47,11.91,34.24,3.21,0.00,Bhopal,Madhya Pradesh,23.2599,77.4126
9,2016-01-10,4.7585,18.55,27.84,10.49,22.46,2.20,0.00,Bhopal,Madhya Pradesh,23.2599,77.4126


In [8]:
print("DATASET VALIDATION REPORT")
print("=" * 45)

print("\nDataset shape:")
print(combined_data.shape)

print("\nCities:")
print(combined_data["CITY"].unique())

print("\nDate range:")
print(
    combined_data["DATE"].min(),
    "to",
    combined_data["DATE"].max()
)

print("\nDuplicate records:")
print(
    combined_data.duplicated(
        subset=["CITY", "DATE"]
    ).sum()
)

print("\nMissing values:")
display(combined_data.isnull().sum())

print("\nStatistical summary:")
display(combined_data.describe())

print("\nRecords per city:")
display(
    combined_data.groupby("CITY").size()
)

DATASET VALIDATION REPORT

Dataset shape:
(36530, 12)

Cities:
<StringArray>
[   'Bhopal',  'Dehradun',  'Gurugram',    'Jaipur',   'Lucknow',  'Ludhiana',
 'New Delhi',     'Patna',    'Shimla',  'Srinagar']
Length: 10, dtype: str

Date range:
2016-01-01 00:00:00 to 2025-12-31 00:00:00

Duplicate records:
0

Missing values:


DATE                 0
ALLSKY_SFC_SW_DWN    0
T2M                  0
T2M_MAX              0
T2M_MIN              0
RH2M                 0
WS10M                0
PRECTOTCORR          0
CITY                 0
STATE                0
LATITUDE             0
LONGITUDE            0
dtype: int64


Statistical summary:


,DATE,ALLSKY_SFC_SW_DWN,T2M,T2M_MAX,T2M_MIN,RH2M,WS10M,PRECTOTCORR,LATITUDE,LONGITUDE
count,36530,36530.000000,36530.000000,36530.000000,36530.000000,36530.000000,36530.000000,36530.000000,36530.000000,36530.00000
mean,2020-12-31 00:00:00,4.792078,22.574614,29.137666,16.979482,54.274312,2.787179,2.859005,28.609250,77.93795
min,2016-01-01 00:00:00,0.257800,-10.750000,-4.350000,-18.970000,5.960000,0.500000,0.000000,23.259900,74.79730
25%,2018-07-02 00:00:00,3.666550,16.430000,24.020000,10.330000,36.630000,2.040000,0.000000,26.846700,75.85730
50%,2020-12-31 00:00:00,4.806000,23.550000,29.710000,17.820000,53.655000,2.560000,0.000000,28.536700,77.19120
75%,2023-07-02 00:00:00,5.990825,29.160000,34.860000,24.550000,73.230000,3.310000,1.690000,30.901000,78.03220
max,2025-12-31 00:00:00,8.986600,42.010000,48.810000,35.750000,96.350000,10.970000,262.610000,34.083700,85.13760
std,NaN,1.597229,8.760418,8.728523,8.786942,21.845505,1.075608,7.735222,2.967855,2.86245



Records per city:


CITY
Bhopal       3653
Dehradun     3653
Gurugram     3653
Jaipur       3653
Lucknow      3653
Ludhiana     3653
New Delhi    3653
Patna        3653
Shimla       3653
Srinagar     3653
dtype: int64